# 章节实践

通过本章的系统学习，我们已经掌握了 regbase 编程模型的核心概念、VF 寄存器操作和常用 VF 指令。为了巩固所学知识，现提供以下实践练习：使用 regbase 模式实现 FastGelu 算子。

**FastGelu 公式**：

$$y = \frac{x}{1 + e^{-x}}$$

要求：
- 编写 VF 函数，实现 FastGelu 的寄存器级计算
- 遵循 Load → Compute → Store → MemoryBarrier 四步流程
- 处理数据量 = 16384 个 float，8 核并行，tileLength = 128
- 使用双缓冲（BUFFER_NUM = 2）

请开始你的实践，体验从理解到创造的完整开发过程。

---

## 环境准备

In [ ]:
!mkdir -p Sources/01.06

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("Environment initialization process completed successfully!")

---

## 头文件和常量定义

编写头文件和常量定义，包括 AscendC 和 Reg 命名空间、核数、数据长度、tiling 结构体等。

In [ ]:
%%writefile Sources/01.06/fast_gelu_practice.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <cmath>
#include "acl/acl.h"
#include "kernel_operator.h"

using namespace AscendC;
using namespace AscendC::Reg;

constexpr uint32_t BUFFER_NUM = 2;
constexpr uint32_t TOTAL_LENGTH = 8 * 2048;
constexpr uint32_t TILE_NUM = 8;
constexpr uint32_t BLOCK_DIM = 8;
constexpr uint32_t TILE_LENGTH = 128;

struct FastGeluTiling {
    uint32_t totalLength;
    uint32_t tileNum;
};

---

## VF 函数实现

实现 FastGelu 的 VF 函数，完成 `y = x / (1 + exp(-1.702 * x))` 的寄存器级计算。

流程说明：
1. `LoadAlign` — 将数据从 UB 加载到寄存器
2. `Neg` — 对 x 取反得到 -x
3. `Exp` — 计算 exp(-x)
4. `Adds` — 加 1.0 得到 1 + exp(-x)
5. `Div` — 执行 x / (1 + exp(-x))
6. `StoreAlign` — 将结果写回 UB
7. `LocalMemBar` — 确保存储可见性

使用 `MaskReg` 和 `RegTensor` 进行寄存器操作，`CreateMask<T, MaskPattern::ALL>()` 创建全掩码。

In [ ]:
%%writefile -a Sources/01.06/fast_gelu_practice.asc

// FastGelu VF: y = x / (1 + exp(-1.702 * x))
template <typename T>
__simd_vf__ inline void FastGeluVF(__ubuf__ T* xAddr, __ubuf__ T* yAddr,
                                     uint32_t vecLen, uint32_t repeat) {
    MaskReg mask = CreateMask<T, MaskPattern::ALL>();
    RegTensor<T> xReg, tReg;

    for (uint16_t i = 0; i < repeat; ++i) {
        LoadAlign(xReg, xAddr + i * vecLen);         // Load: UB -> Reg
        Neg(tReg, xReg, mask);                       // t = -x
        Exp(tReg, tReg, mask);                       // t = exp(-x)
        Adds(tReg, tReg, 1.0f, mask);                // t = 1 + exp(-x)
        Div(tReg, xReg, tReg, mask);                 // t = x / t
        StoreAlign(yAddr + i * vecLen, tReg, mask);  // Store: Reg -> UB
    }
    LocalMemBar<MemType::VEC_STORE, MemType::VEC_LOAD>();
}

---

## Kernel 实现

实现 KernelFastGelu 类，采用 CopyIn / Compute / CopyOut 三段式流水线：

- **Init**：分核逻辑，计算当前核的数据偏移（blockLength * GetBlockIdx()），初始化队列缓冲区
- **CopyIn**：AllocTensor → DataCopy → EnQue，从 Global Memory 搬运到 Local Memory
- **Compute**：DeQue → 调用 FastGeluVF → EnQue → FreeTensor，执行 VF 寄存器计算
- **CopyOut**：DeQue → DataCopy → FreeTensor，将结果写回 Global Memory

使用双缓冲（BUFFER_NUM = 2）实现流水线并行，提升计算效率。

In [ ]:
%%writefile -a Sources/01.06/fast_gelu_practice.asc

class KernelFastGelu {
public:
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y) {
        blockLength = TOTAL_LENGTH / BLOCK_DIM;
        tileLength = blockLength / TILE_NUM / BUFFER_NUM;
        vecLen = GetVecLen() / sizeof(float);
        uint32_t offset = blockLength * GetBlockIdx();
        xGm.SetGlobalBuffer((__gm__ float*)x + offset, blockLength);
        yGm.SetGlobalBuffer((__gm__ float*)y + offset, blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, tileLength * sizeof(float));
        pipe.InitBuffer(outQueueY, BUFFER_NUM, tileLength * sizeof(float));
    }

    __aicore__ inline void Process() {
        for (int32_t i = 0; i < TILE_NUM * BUFFER_NUM; i++) {
            CopyIn(i); Compute(i); CopyOut(i);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress) {
        LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
        DataCopy(xLocal, xGm[progress * tileLength], tileLength);
        inQueueX.EnQue(xLocal);
    }

    __aicore__ inline void Compute(int32_t progress) {
        LocalTensor<float> xLocal = inQueueX.DeQue<float>();
        LocalTensor<float> yLocal = outQueueY.AllocTensor<float>();
        FastGeluVF<float>(
            reinterpret_cast<__ubuf__ float*>(xLocal.GetPhyAddr()),
            reinterpret_cast<__ubuf__ float*>(yLocal.GetPhyAddr()),
            vecLen, tileLength / vecLen);
        outQueueY.EnQue<float>(yLocal);
        inQueueX.FreeTensor(xLocal);
    }

    __aicore__ inline void CopyOut(int32_t progress) {
        LocalTensor<float> yLocal = outQueueY.DeQue<float>();
        DataCopy(yGm[progress * tileLength], yLocal, tileLength);
        outQueueY.FreeTensor(yLocal);
    }

    TPipe pipe;
    TQue<TPosition::VECIN, BUFFER_NUM> inQueueX;
    TQue<TPosition::VECOUT, BUFFER_NUM> outQueueY;
    GlobalTensor<float> xGm, yGm;
    uint32_t blockLength, tileLength, vecLen;
};

__vector__ __global__ __aicore__ void fast_gelu_kernel(GM_ADDR x, GM_ADDR y,
                                              FastGeluTiling tiling) {
    KernelFastGelu op;
    op.Init(x, y);
    op.Process();
}

---

## Host 主函数

实现 Host 侧主函数，包含以下步骤：
1. ACL 初始化（aclInit, aclrtSetDevice, aclrtCreateStream）
2. 分配 Host 和设备内存（aclrtMallocHost, aclrtMalloc）
3. 初始化输入数据（x 在 [-4.0, +4.0] 区间等间隔采样）
4. 将数据拷贝到设备
5. 调用核函数 fast_gelu_kernel，8 核并行
6. 将结果拷贝回 Host
7. 与 CPU 计算结果对比验证（误差阈值 1e-4）
8. 输出 PASSED / FAILED
9. 资源清理

In [ ]:
%%writefile -a Sources/01.06/fast_gelu_practice.asc

int32_t main() {
    aclInit(nullptr);
    aclrtSetDevice(0);
    aclrtStream stream;
    aclrtCreateStream(&stream);

    float *xHost, *yHost;
    aclrtMallocHost((void**)&xHost, TOTAL_LENGTH * sizeof(float));
    aclrtMallocHost((void**)&yHost, TOTAL_LENGTH * sizeof(float));
    for (uint32_t i = 0; i < TOTAL_LENGTH; i++) {
        xHost[i] = -4.0f + i * 8.0f / TOTAL_LENGTH;
    }

    void *xDev, *yDev;
    aclrtMalloc(&xDev, TOTAL_LENGTH * sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc(&yDev, TOTAL_LENGTH * sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(xDev, TOTAL_LENGTH * sizeof(float), xHost,
                TOTAL_LENGTH * sizeof(float), ACL_MEMCPY_HOST_TO_DEVICE);

    FastGeluTiling tiling = {TOTAL_LENGTH, TILE_NUM};
    fast_gelu_kernel<<<BLOCK_DIM, nullptr, stream>>>((uint8_t*)xDev, (uint8_t*)yDev, tiling);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(yHost, TOTAL_LENGTH * sizeof(float), yDev,
                TOTAL_LENGTH * sizeof(float), ACL_MEMCPY_DEVICE_TO_HOST);

    bool pass = true;
    for (uint32_t i = 0; i < TOTAL_LENGTH; i++) {
        if (fabsf(yHost[i] - xHost[i] / (1.0f + expf(-xHost[i]))) > 1e-4f) {
            pass = false; break;
        }
    }
    printf("regbase FastGelu: %s\n", pass ? "PASSED" : "FAILED");

    aclrtFree(xDev); aclrtFree(yDev);
    aclrtFreeHost(xHost); aclrtFreeHost(yHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(0);
    aclFinalize();
    return 0;
}

---

## 编译运行

使用 bisheng 编译工具编译 .asc 源文件，生成可执行文件。

In [ ]:
!bisheng Sources/01.06/fast_gelu_practice.asc --npu-arch=dav-3510 -o Sources/01.06/fast_gelu_practice -lm && echo "Build Successful"

In [ ]:
!./Sources/01.06/fast_gelu_practice

预期输出：`regbase FastGelu: PASSED`

---

## 答案参考

完成实践后，可以参考以下答案代码。

In [ ]:
!cat ./answer/01.06_answer/fast_gelu_practice.asc